# Build the background-flow cache

Run this notebook once before `background_relative_beta_effect.ipynb`. It uses the current Jupyter kernel, avoiding differences between the notebook environment and the shell Python.

The builder is restartable: completed monthly-file partitions are skipped if the notebook is interrupted and rerun.

In [1]:
# Main user settings
N_WORKERS = 4       # Start with 4; more workers may become I/O-limited
DEPTH_MAX_M = 500.0
ANNULUS_INNER_RC = 1.5
ANNULUS_OUTER_RC = 3.0

In [2]:
from pathlib import Path
import sys

HERE = Path.cwd()
if not (HERE / 'background_flow_tools.py').exists():
    HERE = Path('MRes/seacofs_eddy_tilt_analysis/beta_effect_background_flow').resolve()
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(HERE.parent))

import seacofs_tilt_tools as tilt
from background_flow_tools import BackgroundConfig, build_background_cache

print('Python:', sys.version.split()[0])
print('Workers:', N_WORKERS)
if sys.version_info < (3, 9):
    raise RuntimeError('Select a Jupyter kernel running Python 3.9 or newer.')

Python: 3.10.8
Workers: 4


## Load and select eddies

The topographic PV-gradient calculation retains `(w + f)`. The cache is built for observations where the planetary PV-gradient magnitude exceeds the topographic magnitude (`topo_plan_ratio < 0`), independently for AEs and CEs.

In [3]:
paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
eddies, _ = tilt.load_tilt_tables(paths)
eddies = tilt.add_pv_gradient_terms(eddies, grid)
# selected = eddies.loc[eddies['topo_plan_ratio'] < 0].copy()
selected = eddies.copy()
print(f"Selected {len(selected):,} observations from {selected['Eddy'].nunique():,} eddies")

Selected 16,783 observations from 1,609 eddies


## Build or resume the cache

This is the long-running cell. It reads each archive file once, accumulates the full monthly climatology, and extracts cropped annulus backgrounds for matching eddy days.

In [ ]:
config = BackgroundConfig(
    depth_max_m=DEPTH_MAX_M,
    annulus_inner_rc=ANNULUS_INNER_RC,
    annulus_outer_rc=ANNULUS_OUTER_RC,
)
background = build_background_cache(
    selected, grid, config=config, workers=N_WORKERS
)
print('Cache complete:', config.background_table_path)
print(f"Rows: {len(background):,}; eddies: {background['Eddy'].nunique():,}")

Reading 30/30 surface sigma levels.
All sigma levels are required because sampled shallow columns lie entirely above the depth limit.


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:   17.1s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:   26.6s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:   42.4s
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:   57.0s


When the final cell finishes, open and run `background_relative_beta_effect.ipynb`. If this notebook is interrupted, rerun it with the same settings; completed monthly files will be reused.